# Pre-Fire Building Footprints — Camp Fire Area (2018)

**Research context:** WUI fire spatial analysis — building footprint geometry, spacing,
and orientation prior to major fire events.

**Strategy:**
1. **Fetch the Camp Fire perimeter** from CAL FIRE / NIFC open data.
2. **OSM via ohsome API** — query as of 2018-11-07 (day before ignition).
3. **Microsoft ML via Overture Maps** — fill gaps where OSM had no coverage.
4. **Clip to fire perimeter** — keep only buildings inside the burned area.
5. **Annotate** every feature with `source` and `source_date`.

**Fire event:** Camp Fire — ignition 2018-11-08 ~06:30 PST  
**Structures destroyed:** ~18,804 across Paradise, Magalia, Concow, and surrounding areas  
**Burn area:** ~153,336 acres (~620 km²)


---
## 0. Setup

```bash
pip install requests geopandas matplotlib folium shapely scipy mercantile
```

In [ ]:
import gzip
import io
import json
import math
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mercantile
import numpy as np
import pandas as pd
import requests
import folium
from shapely.geometry import Polygon, MultiPolygon, shape, mapping
from shapely.ops import unary_union
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

DATA_DIR = Path('.')   # save outputs alongside this notebook
print('Setup complete.')

---
## 1. Study Area

In [ ]:
# The bbox is derived from the Camp Fire perimeter (fetched next cell).
# We define a conservative outer bbox here to drive the ohsome / Overture queries;
# actual clipping to the burn perimeter happens after the merge step.
#
# Camp Fire affected communities: Paradise, Magalia, Concow, Pines, Yankee Hill
BBOX = (-121.80, 39.65, -121.30, 40.00)   # conservative outer envelope
xmin, ymin, xmax, ymax = BBOX

OSM_DATE = "2018-11-07T23:59:59Z"

print(f"Outer query envelope : {BBOX}")
print(f"  Width  : ~{(xmax-xmin)*111:.0f} km")
print(f"  Height : ~{(ymax-ymin)*111:.0f} km")
print(f"OSM date : {OSM_DATE}")
print()
print("NOTE: Buildings will be clipped to the actual fire perimeter after the merge.")
print("Expected run time: ohsome ~3-5 min, Overture ~2-3 min for this area.")


In [ ]:
# Fetch the Camp Fire perimeter polygon from CAL FIRE open data (FRAP).
# Falls back to NIFC, then to a simplified hardcoded polygon.

CALFIRE_URL = (
    "https://services1.arcgis.com/jUJYIo9tSA7EHvfZ/arcgis/rest/services/"
    "California_Fire_Perimeters_all/FeatureServer/0/query"
)
NIFC_URL = (
    "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/"
    "WFIGS_Interagency_Perimeters/FeatureServer/0/query"
)

fire_perimeter = None

# --- Try CAL FIRE first ---
try:
    print("Fetching Camp Fire perimeter from CAL FIRE...")
    r = requests.get(CALFIRE_URL, params={
        'where': "FIRE_NAME='CAMP' AND YEAR_=2018",
        'outFields': 'FIRE_NAME,YEAR_,GIS_ACRES,ALARM_DATE',
        'f': 'geojson', 'outSR': '4326',
    }, timeout=60)
    r.raise_for_status()
    fc = r.json()
    feats = fc.get('features', [])
    if feats:
        # Pick the largest polygon (final perimeter)
        geoms = [shape(f['geometry']) for f in feats if f.get('geometry')]
        fire_perimeter = max(geoms, key=lambda g: g.area)
        print(f"  ✓ CAL FIRE: {len(feats)} feature(s), using largest polygon")
    else:
        print("  No features returned from CAL FIRE")
except Exception as e:
    print(f"  CAL FIRE failed: {e}")

# --- Try NIFC if CAL FIRE failed ---
if fire_perimeter is None:
    try:
        print("Trying NIFC...")
        r = requests.get(NIFC_URL, params={
            'where': "attr_IncidentName='CAMP' AND attr_FireDiscoveryDateTime >= '2018-11-07'  AND attr_FireDiscoveryDateTime <= '2018-11-10'",
            'outFields': '*', 'f': 'geojson', 'outSR': '4326',
        }, timeout=60)
        r.raise_for_status()
        fc = r.json()
        feats = fc.get('features', [])
        if feats:
            geoms = [shape(f['geometry']) for f in feats if f.get('geometry')]
            fire_perimeter = max(geoms, key=lambda g: g.area)
            print(f"  ✓ NIFC: {len(feats)} feature(s)")
        else:
            print("  No features returned from NIFC")
    except Exception as e:
        print(f"  NIFC failed: {e}")

# --- Hardcoded fallback: simplified Camp Fire burn envelope ---
if fire_perimeter is None:
    print("Using hardcoded Camp Fire approximate perimeter as fallback.")
    from shapely.geometry import box
    # Conservative bounding polygon of the burn area
    fire_perimeter = box(-121.77, 39.68, -121.33, 39.97)

# Tighten the query bbox to the perimeter bounds
pb = fire_perimeter.bounds   # (minx, miny, maxx, maxy)
# Add a small buffer (~500m) so we catch buildings just outside the perimeter edge
BBOX = (pb[0] - 0.005, pb[1] - 0.005, pb[2] + 0.005, pb[3] + 0.005)
xmin, ymin, xmax, ymax = BBOX

print(f"\nFire perimeter area : {fire_perimeter.area * 111**2:.0f} km²  "
      f"({fire_perimeter.area * 111**2 * 247:.0f} acres)")
print(f"Query bbox          : {BBOX}")

# Quick static plot of the perimeter
fig, ax = plt.subplots(figsize=(7, 6))
gpd.GeoSeries([fire_perimeter]).plot(ax=ax, facecolor='#FF6B3520',
                                     edgecolor='#FF6B35', linewidth=1.5)
ax.set_title('Camp Fire Perimeter — Query Area', fontsize=12)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(DATA_DIR / 'camp_fire_perimeter.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 2. Source 1 — OSM via ohsome API

The **[ohsome API](https://api.ohsome.org)** (HeiGIT / Heidelberg University) is purpose-built for
historical OSM analysis. It stores the full OSM edit history and reconstructs any past snapshot
on demand — far more reliable than the Overpass attic feature, which frequently times out or
rejects historical queries.

- Endpoint: `POST /elements/geometry`
- Key parameter: `time=2018-11-07` — returns the OSM state exactly on that date
- Response: proper GeoJSON with `@timestamp` per feature (the edit date of the active version)
- Filter: `building=* and (geometry:polygon or geometry:multipolygon)`

No API key required. Rate-limited but generous for research use.

In [ ]:
OHSOME_URL = "https://api.ohsome.org/v1/elements/geometry"

OHSOME_BBOX   = f"{xmin},{ymin},{xmax},{ymax}"
OHSOME_PARAMS = {
    "bboxes":     OHSOME_BBOX,
    "filter":     "building=* and geometry:polygon",
    "time":       OSM_DATE[:10],
    "properties": "tags,metadata",
}

print(f"Querying ohsome API...")
print(f"  bbox   : {OHSOME_BBOX}")
print(f"  time   : {OHSOME_PARAMS['time']}")
print(f"  filter : {OHSOME_PARAMS['filter']}")

resp = requests.post(OHSOME_URL, data=OHSOME_PARAMS, timeout=300)
if resp.status_code != 200:
    print(f"ERROR {resp.status_code}: {resp.text[:800]}")
    raise RuntimeError(f"ohsome API returned {resp.status_code}")

osm_fc = resp.json()
n_feat = len(osm_fc.get('features', []))
print(f"  ✓  {n_feat:,} features returned")

# ── Inspect property keys from the first feature ──────────────────────────
if n_feat > 0:
    sample_props = osm_fc['features'][0].get('properties', {})
    print("\nProperty keys in first feature:")
    for k, v in sample_props.items():
        print(f"  {k!r:35s} : {str(v)[:60]}")

    # Auto-detect the timestamp field
    TIMESTAMP_CANDIDATES = ['@timestamp', '@snapshotTimestamp', '@validFrom',
                             'timestamp', '@changesetTimestamp']
    TIMESTAMP_KEY = next(
        (k for k in TIMESTAMP_CANDIDATES if k in sample_props), None
    )
    print(f"\nAuto-detected timestamp field: {TIMESTAMP_KEY!r}")
else:
    TIMESTAMP_KEY = '@timestamp'
    print('No features — cannot detect timestamp field.')

# ── Parse ─────────────────────────────────────────────────────────────────
def parse_ohsome_buildings(fc, ts_key):
    """Parse ohsome GeoJSON into a GeoDataFrame.
    ts_key: the property name for the element timestamp (auto-detected above).
    """
    records = []
    skipped = 0

    for feat in fc.get('features', []):
        props    = feat.get('properties', {}) or {}
        geom_raw = feat.get('geometry')

        if geom_raw is None:
            skipped += 1
            continue
        try:
            geom = shape(geom_raw)
        except Exception:
            skipped += 1
            continue

        if not geom.is_valid:
            geom = geom.buffer(0)
        if geom.is_empty or not geom.is_valid:
            skipped += 1
            continue

        osm_id_full = props.get('@osmId', '')
        osm_type    = props.get('@osmType', '')
        osm_id_num  = osm_id_full.split('/')[-1] if '/' in osm_id_full else osm_id_full

        records.append({
            'osm_id':           osm_id_num,
            'osm_type':         osm_type,
            'geometry':         geom,
            'source':           'openstreetmap',
            'source_date':      props.get(ts_key) if ts_key else None,
            'osm_version':      props.get('@version'),
            'building':         props.get('building', 'yes'),
            'name':             props.get('name'),
            'height':           props.get('height'),
            'levels':           props.get('building:levels'),
            'addr_street':      props.get('addr:street'),
            'addr_housenumber': props.get('addr:housenumber'),
        })

    print(f"Parsed {len(records):,} valid building geometries  ({skipped} skipped)")
    return gpd.GeoDataFrame(records, crs='EPSG:4326')


gdf_osm = parse_ohsome_buildings(osm_fc, TIMESTAMP_KEY)
gdf_osm['source_date'] = pd.to_datetime(gdf_osm['source_date'], utc=True, errors='coerce')

has_dates = gdf_osm['source_date'].notna().sum()
print(f"\nBuildings with timestamp : {has_dates:,} / {len(gdf_osm):,}")
if has_dates > 0:
    print(f"Date range : {gdf_osm['source_date'].min().date()} → {gdf_osm['source_date'].max().date()}")
print()
print(gdf_osm[['osm_id', 'osm_type', 'source_date', 'building']].head(8))


---
## 3. Source 2 — Microsoft ML Buildings via Overture Maps

Microsoft's Global ML Building Footprints are one of the source datasets that Overture Maps
ingests and conflates. Rather than downloading Microsoft's tiles directly (their blob storage
endpoint recently restricted public access), we query Overture's current release and keep only
buildings whose primary source is Microsoft.

**Why this works for pre-fire gap-filling:** Buildings that appear in Microsoft's ML dataset
but were absent from OSM in 2018 almost certainly existed physically before the fire —
they simply weren't community-mapped yet. We're using ML footprints to fill spatial gaps,
not to make temporal claims.

**Note:** The Overture download requires the `pyarrow` package and anonymous S3 access
to `overturemaps-us-west-2` (us-west-2 region).


In [ ]:
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.fs as fs
import pyarrow.parquet as pq
from shapely import wkb
from urllib.request import urlopen

# Discover the latest Overture release from the STAC catalog
with urlopen('https://stac.overturemaps.org/catalog.json') as r:
    catalog = json.load(r)
LATEST_RELEASE = catalog['latest']
print(f"Latest Overture release: {LATEST_RELEASE}")

S3_PATH = f"overturemaps-us-west-2/release/{LATEST_RELEASE}/theme=buildings/type=building/"
s3 = fs.S3FileSystem(anonymous=True, region='us-west-2')
ov_dataset = ds.dataset(S3_PATH, filesystem=s3)

# Spatial filter using bbox column (efficient row-group pruning)
bbox_filter = (
    (pc.field('bbox', 'xmin') < xmax) &
    (pc.field('bbox', 'xmax') > xmin) &
    (pc.field('bbox', 'ymin') < ymax) &
    (pc.field('bbox', 'ymax') > ymin)
)

print("Downloading Overture buildings for Paradise bbox...")
ov_table = ov_dataset.to_table(filter=bbox_filter)
print(f"  ✓  {ov_table.num_rows:,} total Overture buildings retrieved")


In [ ]:
# Parse Overture rows — keep ALL non-OSM sources
# Overture conflates: Microsoft ML, Google Open Buildings, Esri, and others.
# Previously we kept only 'microsoft'; now we keep everything that isn't OSM
# (OSM is already captured via the ohsome query with precise dating).
ml_records = []
source_tally = {}

for row in ov_table.to_pylist():
    sources = row.get('sources') or []
    if not sources:
        continue

    primary = sources[0].get('dataset', '').lower()

    # Skip OSM-primary buildings — ohsome already gave us those with dates
    if 'openstreetmap' in primary:
        continue

    try:
        geom = wkb.loads(bytes(row['geometry']))
    except Exception:
        continue
    if geom is None or geom.is_empty:
        continue
    if not geom.is_valid:
        geom = geom.buffer(0)

    ml_records.append({
        'geometry':      geom,
        'ml_source':     primary,          # e.g. 'microsoft-buildings', 'google-open-buildings'
        'ms_confidence': sources[0].get('confidence'),
        'overture_id':   row.get('id'),
    })
    source_tally[primary] = source_tally.get(primary, 0) + 1

print(f"Non-OSM Overture buildings in study area: {len(ml_records):,}")
print("\nBreakdown by ML source:")
for src, cnt in sorted(source_tally.items(), key=lambda x: -x[1]):
    print(f"  {src:<45} {cnt:>6,}")


In [ ]:
# Standardise ML GDF — rename ms_records → ml_records
gdf_ms_raw = gpd.GeoDataFrame(ml_records, crs='EPSG:4326')
gdf_ms_raw = gdf_ms_raw[gdf_ms_raw.geometry.notna() & ~gdf_ms_raw.geometry.is_empty].copy()


In [ ]:
gdf_ms = gpd.GeoDataFrame({
    'osm_id':           None,
    'osm_type':         None,
    'geometry':         gdf_ms_raw.geometry,
    # Use actual source name from Overture (microsoft-ml, google-open-buildings, etc.)
    'source':           gdf_ms_raw['ml_source'].values,
    'source_date':      pd.NaT,
    'osm_version':      None,
    'building':         'yes',
    'name':             None,
    'height':           None,
    'levels':           None,
    'addr_street':      None,
    'addr_housenumber': None,
}, crs='EPSG:4326')

if 'ms_confidence' in gdf_ms_raw.columns:
    gdf_ms['ms_confidence'] = gdf_ms_raw['ms_confidence'].values

print(f"ML buildings (all non-OSM Overture sources): {len(gdf_ms):,}")


---
## 4. Merge: OSM Priority, Microsoft Gap-Fill

We keep all OSM buildings. For Microsoft buildings, we drop any that overlap
substantially with an existing OSM footprint (IoU > 0.3 threshold).
This prevents double-counting while retaining genuine Microsoft-only structures.

In [ ]:
def remove_ms_duplicates(gdf_osm, gdf_ms, iou_threshold=0.3):
    """Drop Microsoft buildings that overlap with OSM buildings above an IoU threshold.
    
    Uses a spatial index for efficiency.
    """
    if len(gdf_ms) == 0 or len(gdf_osm) == 0:
        return gdf_ms

    osm_sindex = gdf_osm.sindex
    keep_mask = np.ones(len(gdf_ms), dtype=bool)

    for i, ms_row in gdf_ms.iterrows():
        ms_geom = ms_row.geometry
        if ms_geom is None or ms_geom.is_empty:
            keep_mask[i] = False
            continue

        # Candidate OSM buildings that might intersect
        candidates = list(osm_sindex.intersection(ms_geom.bounds))
        if not candidates:
            continue

        for j in candidates:
            osm_geom = gdf_osm.iloc[j].geometry
            if not ms_geom.intersects(osm_geom):
                continue
            intersection = ms_geom.intersection(osm_geom).area
            union = ms_geom.union(osm_geom).area
            if union > 0 and (intersection / union) >= iou_threshold:
                keep_mask[i] = False
                break

    return gdf_ms[keep_mask].reset_index(drop=True)


print(f"OSM buildings:       {len(gdf_osm):,}")
print(f"Microsoft buildings: {len(gdf_ms):,}")
print("Removing Microsoft duplicates (IoU > 0.3 with any OSM footprint)...")

gdf_ms_deduped = remove_ms_duplicates(gdf_osm, gdf_ms, iou_threshold=0.3)
n_dropped = len(gdf_ms) - len(gdf_ms_deduped)

print(f"  Dropped {n_dropped:,} Microsoft duplicates")
print(f"  Kept {len(gdf_ms_deduped):,} Microsoft gap-fill buildings")

In [ ]:
# Concatenate final dataset
gdf_all = pd.concat([gdf_osm, gdf_ms_deduped], ignore_index=True)
gdf_all = gpd.GeoDataFrame(gdf_all, crs='EPSG:4326')

# Ensure valid geometries only
gdf_all = gdf_all[gdf_all.geometry.notna() & ~gdf_all.geometry.is_empty].copy()
gdf_all = gdf_all[gdf_all.geometry.is_valid].copy()

print(f"\n=== MERGED DATASET ===")
print(f"Total buildings:  {len(gdf_all):,}")
print()
print(gdf_all['source'].value_counts().to_string())
print()
print(f"Buildings with source_date:    {gdf_all['source_date'].notna().sum():,}")
print(f"Buildings without source_date: {gdf_all['source_date'].isna().sum():,}")

In [ ]:
# Clip merged buildings to the actual fire perimeter
# Buildings outside the burn area are irrelevant to WUI spread analysis
print(f"Before clip : {len(gdf_all):,} buildings")

fire_gdf = gpd.GeoDataFrame(geometry=[fire_perimeter], crs='EPSG:4326')
gdf_all  = gpd.clip(gdf_all, fire_gdf).reset_index(drop=True)

print(f"After clip  : {len(gdf_all):,} buildings (within Camp Fire perimeter)")
print()
print(gdf_all['source'].value_counts().to_string())


---
## 5. Temporal Annotation

OSM buildings carry their last-edit timestamp. We inspect the date distribution
relative to the fire to understand data currency.

In [ ]:
FIRE_DT = pd.Timestamp('2018-11-08', tz='UTC')

# Re-cast after concat — pd.concat can demote mixed datetime/NaT to object dtype
gdf_all['source_date'] = pd.to_datetime(gdf_all['source_date'], utc=True, errors='coerce')

osm_dated = gdf_all[
    (gdf_all['source'] == 'openstreetmap') & gdf_all['source_date'].notna()
].copy()

print(f"OSM buildings total        : {(gdf_all['source'] == 'openstreetmap').sum():,}")
print(f"OSM buildings with date    : {len(osm_dated):,}")
print(f"OSM buildings without date : {(gdf_all['source'] == 'openstreetmap').sum() - len(osm_dated):,}")
print()

if len(osm_dated) == 0:
    print('WARNING: No dated OSM buildings found.')
    print('Sample source_date values from gdf_osm:')
    print(gdf_osm['source_date'].head(10).tolist())
else:
    osm_dated['edit_year'] = osm_dated['source_date'].dt.year
    print(f"Date range : {osm_dated['source_date'].min().date()} → {osm_dated['source_date'].max().date()}")
    print(f"All pre-fire? {(osm_dated['source_date'] < FIRE_DT).all()}")
    print()

    year_counts = osm_dated['edit_year'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.bar(year_counts.index, year_counts.values,
           color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Year of last OSM edit', fontsize=11)
    ax.set_ylabel('Number of buildings', fontsize=11)
    ax.set_title('OSM Building Edit Year Distribution — Paradise, CA\n'
                 '(queried as of 2018-11-07; all dates are pre-fire by definition)', fontsize=12)
    ax.set_xticks(year_counts.index)
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'paradise_osm_edit_years.png', dpi=150, bbox_inches='tight')
    plt.show()


---
## 6. Visualise — Source Map

In [ ]:
SOURCE_COLORS = {
    'openstreetmap':           '#2196F3',   # blue
    'microsoft-buildings':     '#FF6B35',   # orange
    'google-open-buildings':   '#4CAF50',   # green
    'esri-buildings':          '#9C27B0',   # purple
}

def get_color(src):
    src = (src or '').lower()
    for key, color in SOURCE_COLORS.items():
        if key in src:
            return color
    return '#9E9E9E'   # grey for unknown

gdf_all['color'] = gdf_all['source'].apply(get_color)

fig, ax = plt.subplots(figsize=(14, 9))
gdf_all.plot(ax=ax, color=gdf_all['color'], edgecolor='none', alpha=0.75)

# Add fire perimeter outline
gpd.GeoSeries([fire_perimeter]).plot(ax=ax, facecolor='none',
                                       edgecolor='red', linewidth=1, linestyle='--')

# Legend — only show sources that appear in the data
import matplotlib.patches as mpatches
present_sources = gdf_all['source'].value_counts()
legend_patches = []
for src, cnt in present_sources.items():
    color = get_color(src)
    legend_patches.append(mpatches.Patch(color=color, label=f"{src} ({cnt:,})"))
legend_patches.append(mpatches.Patch(facecolor='none', edgecolor='red',
                                       linestyle='--', label='Camp Fire perimeter'))
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

ax.set_title('Pre-Fire Building Footprints — Camp Fire Area\n'
             'OSM as of 2018-11-07 + ML gap-fill (clipped to CAL FIRE burn perimeter)',
             fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(DATA_DIR / 'camp_fire_prefire_source_map.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Interactive folium map
m = folium.Map(
    location=[(ymin + ymax) / 2, (xmin + xmax) / 2],
    zoom_start=14,
    tiles='CartoDB dark_matter'
)

for _, row in gdf_all.iterrows():
    try:
        date_str = row['source_date'].strftime('%Y-%m-%d') if pd.notna(row['source_date']) else 'N/A'
        folium.GeoJson(
            row['geometry'].__geo_interface__,
            style_function=lambda x, c=row['color']: {
                'fillColor': c, 'color': c,
                'weight': 0.4, 'fillOpacity': 0.65
            },
            tooltip=folium.Tooltip(
                f"<b>Source:</b> {row['source']}<br>"
                f"<b>Last edit:</b> {date_str}<br>"
                f"<b>Building:</b> {row.get('building','yes')}"
            )
        ).add_to(m)
    except Exception:
        pass

map_path = DATA_DIR / 'paradise_prefire_interactive.html'
m.save(map_path)
print(f"Interactive map saved → {map_path}")
m

---
## 7. Spatial Metrics for Fire Spread Analysis

Key geometric properties relevant to structure-to-structure fire ignition:
building **orientation** (affects ember exposure surface area) and
**inter-building separation** (governs radiant heat flux and brand ignition probability).

**Two separation metrics are computed:**
- `min_wall_wall_m` — minimum exterior gap between polygon walls (physically meaningful:
  governs radiant heat transfer; used as the primary distance input into SSDD)
- `nn_dist_m` — centroid-to-centroid distance (kept as reference; used by some legacy metrics)

Reference thresholds from WUI fire research:
- **< 3 m**: structures nearly touching — near-certain fire jump
- **< 7.6 m** (~1.5× typical building width): high ignition probability from radiant heat
- **< 15 m**: elevated risk zone for direct flame impingement


In [ ]:
# Project to UTM Zone 10N for accurate metric calculations
gdf_proj = gdf_all.to_crs('EPSG:32610')
gdf_proj['area_m2'] = gdf_proj.geometry.area

# Filter out very small polygons (noise / garages / sheds < 10m²)
gdf_main = gdf_proj[gdf_proj['area_m2'] >= 15].copy()

print(f"Buildings for spatial analysis: {len(gdf_main):,}  (area ≥ 15m²)")
print(f"  Median footprint : {gdf_main['area_m2'].median():.0f} m²")
print(f"  Mean footprint   : {gdf_main['area_m2'].mean():.0f} m²")
print(f"  IQR (25–75%)     : {gdf_main['area_m2'].quantile(0.25):.0f} – "
      f"{gdf_main['area_m2'].quantile(0.75):.0f} m²")

In [ ]:
# Minimum rotated rectangle orientation (0–180°)
def mbr_orientation(geom):
    """Return the long-axis orientation angle (degrees, 0–180) of a polygon."""
    try:
        mbr    = geom.minimum_rotated_rectangle
        coords = list(mbr.exterior.coords)
        edges  = []
        for i in range(len(coords) - 1):
            dx = coords[i+1][0] - coords[i][0]
            dy = coords[i+1][1] - coords[i][1]
            edges.append((math.hypot(dx, dy), dx, dy))
        edges.sort(reverse=True)
        _, dx, dy = edges[0]
        return math.degrees(math.atan2(dy, dx)) % 180
    except Exception:
        return np.nan

gdf_main['orientation_deg'] = gdf_main.geometry.apply(mbr_orientation)

angles = gdf_main['orientation_deg'].dropna().values
bins   = np.linspace(0, 180, 37)   # 5° bins
counts, _ = np.histogram(angles, bins=bins)

# Build figure with one regular axis and one polar axis
fig = plt.figure(figsize=(13, 5))
ax1     = fig.add_subplot(121)
ax_polar = fig.add_subplot(122, projection='polar')

ax1.bar(bins[:-1], counts, width=4.5, color='steelblue', edgecolor='white', alpha=0.85)
ax1.set_xlabel('Long-axis orientation (°)', fontsize=11)
ax1.set_ylabel('Building count', fontsize=11)
ax1.set_title('Building Orientation Distribution\nParadise, CA — pre-fire', fontsize=11)
ax1.set_xlim(0, 180)

# Polar rose — doubled for 0–360 symmetry (orientation is axis, not direction)
theta = np.deg2rad(np.concatenate([bins[:-1], bins[:-1] + 180]))
radii = np.concatenate([counts, counts])
ax_polar.bar(theta, radii, width=np.deg2rad(5), color='steelblue', alpha=0.75)
ax_polar.set_theta_zero_location('N')
ax_polar.set_theta_direction(-1)
ax_polar.set_title('Rose Diagram', pad=15)

plt.tight_layout()
plt.savefig(DATA_DIR / 'paradise_orientation.png', dpi=150, bbox_inches='tight')
plt.show()

dominant = bins[np.argmax(counts)]
print(f"Dominant orientation bin: {dominant:.0f}–{dominant+5:.0f}°")


In [ ]:
# ── Wall-to-wall separation + centroid-to-centroid (for comparison) ──────
# Shapely polygon.distance(other_polygon) returns the minimum gap between
# exterior rings — true wall-to-wall clearance, which is what governs
# radiant heat flux and ember ignition probability in WUI fire spread.
#
# We compute BOTH metrics:
#   min_wall_wall_m  — smallest exterior gap to any non-overlapping neighbour
#   nn_dist_m        — centroid-to-centroid (kept for legacy / SSDD input)
#
# Uses an STRtree for spatial indexing so we only compute polygon.distance()
# against plausible candidates (those whose bounding boxes overlap a buffer).

from shapely.strtree import STRtree
from tqdm.auto import tqdm

WALL_SEARCH_R = 60.0   # metres — only look for neighbours within this buffer
                        # (covers >99% of meaningful fire spread distances)

polys_arr = gdf_main.geometry.values          # numpy array of shapely Polygons
pts_arr   = np.array([[g.centroid.x, g.centroid.y] for g in polys_arr])
tree_poly = STRtree(polys_arr)

min_wall_wall = np.full(len(gdf_main), np.inf)
nn_centroid   = np.full(len(gdf_main), np.inf)

for i in tqdm(range(len(gdf_main)), desc="Wall-to-wall gaps"):
    Pi = polys_arr[i]
    ci = pts_arr[i]

    # Candidate polygons within the search buffer (bounding-box prefilter)
    cand_idxs = tree_poly.query(Pi.buffer(WALL_SEARCH_R))

    for j in cand_idxs:
        if j == i:
            continue

        # Wall-to-wall: minimum distance between polygon exteriors
        # Returns 0.0 if polygons overlap — treat overlapping as touching
        d_wall = Pi.distance(polys_arr[j])
        if d_wall < min_wall_wall[i]:
            min_wall_wall[i] = d_wall

        # Centroid-to-centroid
        d_cen = float(np.hypot(ci[0] - pts_arr[j][0], ci[1] - pts_arr[j][1]))
        if d_cen < nn_centroid[i]:
            nn_centroid[i] = d_cen

# Buildings with no neighbour within WALL_SEARCH_R get NaN (isolated)
min_wall_wall[np.isinf(min_wall_wall)] = np.nan
nn_centroid[np.isinf(nn_centroid)]     = np.nan

gdf_main = gdf_main.copy()
gdf_main['min_wall_wall_m'] = min_wall_wall
gdf_main['nn_dist_m']       = nn_centroid

print(f"Buildings analysed: {len(gdf_main):,}  (search radius {WALL_SEARCH_R} m)")
print()
print("=== Wall-to-Wall Separation (m) ===")
ww = gdf_main['min_wall_wall_m'].dropna()
for p in [5, 10, 25, 50, 75, 90]:
    print(f"  {p:3d}th pct : {np.percentile(ww, p):.1f} m")
print(f"  < 3 m (high ember risk)   : {(ww < 3).sum():,}  ({(ww < 3).mean()*100:.1f}%)")
print(f"  < 7.6 m (1.5× bldg width) : {(ww < 7.6).sum():,}  ({(ww < 7.6).mean()*100:.1f}%)")
print(f"  < 15 m (radiant threshold) : {(ww < 15).sum():,}  ({(ww < 15).mean()*100:.1f}%)")
print()
print("=== Centroid-to-Centroid (m, for reference) ===")
cc = gdf_main['nn_dist_m'].dropna()
for p in [10, 25, 50, 75, 90]:
    print(f"  {p:3d}th pct : {np.percentile(cc, p):.1f} m")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(ww[ww < 60], bins=60, color='#E53935', edgecolor='white', alpha=0.85)
axes[0].axvline(np.median(ww), color='darkred', linestyle='--', linewidth=1.5,
                label=f'Median: {np.median(ww):.1f} m')
axes[0].axvline(7.6,  color='orange', linestyle=':', linewidth=1.5, label='7.6 m (1.5× width)')
axes[0].axvline(15.0, color='gold',   linestyle=':', linewidth=1.5, label='15 m (radiant)')
axes[0].set_xlabel('Min wall-to-wall gap (m)', fontsize=10)
axes[0].set_ylabel('Count', fontsize=10)
axes[0].set_title('Wall-to-Wall Separation — Paradise, CA (clipped 60 m)', fontsize=10)
axes[0].legend(fontsize=8)

axes[1].hist(cc[cc < 120], bins=60, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(np.median(cc), color='darkred', linestyle='--', linewidth=1.5,
                label=f'Median: {np.median(cc):.1f} m')
axes[1].set_xlabel('Centroid-to-centroid distance (m)', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('Centroid-to-Centroid Spacing (reference only)', fontsize=10)
axes[1].legend(fontsize=8)

plt.suptitle('Building Separation — Paradise, CA (pre-fire)', fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / 'paradise_spacing_wall_to_wall.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 9. DINS Validation

CAL FIRE Damage Inspection (DINS) records provide ground-truth structure locations
from post-fire field surveys. We use them to assess how well our footprint dataset
covers the structures that were actually present before the fire.

**DINS file:** `Camp_Test/DINS_Camp.parquet`  
**Strategy:** buffer each DINS point by 20 m and spatially join to footprint polygons.
Any DINS point with no matching footprint is "unmatched" — a gap in our dataset.

We exclude minor/infrastructure/agriculture structure classes (barns, sheds, bridges)
and focus on primary residential and commercial structures.


In [ ]:
# Load CAL FIRE DINS damage inspection points
dins_path = Path('Camp_Test/DINS_Camp.parquet')
gdf_dins_raw = gpd.read_parquet(dins_path)

print(f"DINS raw shape : {gdf_dins_raw.shape}")
print(f"Columns        : {list(gdf_dins_raw.columns)}")
print()
print(gdf_dins_raw[['STRUCTUREC', 'STRUCTURET', 'DAMAGE', 'LATITUDE', 'LONGITUDE']].head(5))


In [ ]:
# Rebuild WGS84 point geometry from LATITUDE / LONGITUDE columns
# (The native geometry column is in a local projected CRS — not WGS84)
from shapely.geometry import Point

EXCLUDE_CLASSES = {'Other Minor Structure', 'Infrastructure', 'Agriculture'}

mask = ~gdf_dins_raw['STRUCTUREC'].isin(EXCLUDE_CLASSES)
dins_primary = gdf_dins_raw[mask].copy()

dins_primary['geometry'] = dins_primary.apply(
    lambda r: Point(r['LONGITUDE'], r['LATITUDE']), axis=1
)
gdf_dins = gpd.GeoDataFrame(dins_primary, geometry='geometry', crs='EPSG:4326')

print(f"Total DINS records          : {len(gdf_dins_raw):,}")
print(f"After excluding minor/infra : {len(gdf_dins):,}")
print()
print("Damage breakdown (primary structures only):")
print(gdf_dins['DAMAGE'].value_counts().to_string())
print()
print("Structure class breakdown:")
print(gdf_dins['STRUCTUREC'].value_counts().to_string())


In [ ]:
# Spatial match: 20 m buffer around each DINS point, join to footprint polygons
# Project both layers to UTM Zone 10N (EPSG:32610) for metric buffering

BUFFER_M = 20

gdf_dins_proj  = gdf_dins.to_crs('EPSG:32610').copy()
gdf_fp_proj    = gdf_all.to_crs('EPSG:32610').copy()

# Buffer DINS points
gdf_dins_proj['geometry_orig'] = gdf_dins_proj.geometry
gdf_dins_proj['geometry']      = gdf_dins_proj.geometry.buffer(BUFFER_M)

# Spatial join — left keeps all DINS, indicator=True shows which matched
joined = gpd.sjoin(
    gdf_dins_proj[['STRUCTUREC', 'STRUCTURET', 'DAMAGE', 'LATITUDE', 'LONGITUDE',
                   'geometry', 'geometry_orig']],
    gdf_fp_proj[['geometry']],
    how='left',
    predicate='intersects'
)

# A DINS point is "matched" if at least one footprint overlaps its buffer
matched_idx   = set(joined[joined['index_right'].notna()].index)
dins_matched  = gdf_dins_proj[gdf_dins_proj.index.isin(matched_idx)].copy()
dins_unmatched= gdf_dins_proj[~gdf_dins_proj.index.isin(matched_idx)].copy()

# Restore original point geometry (not buffered) for display/save
dins_matched['geometry']   = dins_matched['geometry_orig']
dins_unmatched['geometry'] = dins_unmatched['geometry_orig']

n_total   = len(gdf_dins)
n_matched = len(dins_matched)
n_miss    = len(dins_unmatched)

print(f"DINS primary structures : {n_total:,}")
print(f"  Matched  (≥1 footprint within {BUFFER_M}m) : {n_matched:,}  "
      f"({n_matched/n_total*100:.1f}%)")
print(f"  Unmatched (no footprint nearby)            : {n_miss:,}  "
      f"({n_miss/n_total*100:.1f}%)")
print()
print("Unmatched — damage breakdown:")
print(dins_unmatched['DAMAGE'].value_counts().to_string())
print()
print("Unmatched — structure class breakdown:")
print(dins_unmatched['STRUCTUREC'].value_counts().to_string())


In [ ]:
# Re-project back to WGS84 for display
dins_matched_wgs   = dins_matched.to_crs('EPSG:4326')
dins_unmatched_wgs = dins_unmatched.to_crs('EPSG:4326')

fig, ax = plt.subplots(figsize=(14, 9))

# Building footprints (light grey backdrop)
gdf_all.plot(ax=ax, color='#AAAAAA', edgecolor='none', alpha=0.4, label='Footprints')

# DINS matched
dins_matched_wgs.plot(ax=ax, color='#2196F3', markersize=2, alpha=0.6,
                       label=f'DINS matched ({len(dins_matched_wgs):,})')

# DINS unmatched
dins_unmatched_wgs.plot(ax=ax, color='#FF1744', markersize=3, alpha=0.75,
                         label=f'DINS unmatched ({len(dins_unmatched_wgs):,})')

# Fire perimeter
gpd.GeoSeries([fire_perimeter]).plot(ax=ax, facecolor='none',
                                      edgecolor='black', linewidth=1, linestyle='--',
                                      label='Camp Fire perimeter')

ax.legend(fontsize=9, loc='upper right')
ax.set_title(
    f'DINS Coverage Validation — Camp Fire\n'
    f'{n_matched:,} matched ({n_matched/n_total*100:.1f}%) · '
    f'{n_miss:,} unmatched ({n_miss/n_total*100:.1f}%) · '
    f'buffer = {BUFFER_M} m',
    fontsize=11
)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(DATA_DIR / 'dins_coverage_map.png', dpi=150, bbox_inches='tight')
plt.show()

# Save unmatched DINS to GeoPackage for further inspection
unmatched_path = DATA_DIR / 'dins_unmatched_primary.gpkg'
save_cols_dins = ['STRUCTUREC', 'STRUCTURET', 'DAMAGE', 'LATITUDE', 'LONGITUDE', 'geometry']
dins_unmatched_wgs[[c for c in save_cols_dins if c in dins_unmatched_wgs.columns]].to_file(
    unmatched_path, driver='GPKG'
)
print(f"Unmatched DINS saved → {unmatched_path}")


---
## 10. DINS-Centric Unified Building Dataset (Paradise Validation)

**Problem identified in Section 9:** Overture ML buildings are derived from *current* satellite
imagery. Because ~14,000 Camp Fire structures were destroyed and never rebuilt, the current ML
model cannot detect them — producing a systematic post-fire survivorship bias (only 39% match rate).

**Solution:** Treat DINS as the authoritative pre-fire structure inventory.

For each of the 17,964 primary DINS structures we attach either:
- A **real polygon footprint** from our dataset where the nearest building centroid is ≤ 25 m away, or
- An **estimated circular footprint** sized from the median area of real matched buildings in the
  same `STRUCTUREC` class (Single Residence, Commercial, etc.).

Every record gets a `geometry_source` flag — `matched_footprint` or `estimated_centroid` — so
downstream analysis can track which geometry is empirical vs. statistical.

> **Generalisation note (Section 11):** Outside California, replace DINS with county parcel /
> tax-assessor data. Parcel records are keyed to the year of construction, not demolition —
> they are pre-fire by definition and available for virtually every US county.


In [ ]:
# ── Project both layers to UTM Zone 10N ──────────────────────────────
gdf_fp_utm   = gdf_all.to_crs('EPSG:32610').copy()
gdf_fp_utm['area_m2'] = gdf_fp_utm.geometry.area
gdf_dins_utm = gdf_dins.to_crs('EPSG:32610').copy()   # gdf_dins from Section 9

MATCH_M = 25   # metres — DINS GPS accuracy + footprint centroid offset

# ── Build KDTree on footprint centroids (fast nearest-neighbour search) ──
fp_centroids = np.array([[g.centroid.x, g.centroid.y]
                          for g in gdf_fp_utm.geometry])
tree_fp = cKDTree(fp_centroids)

dins_pts = np.array([[g.x, g.y] for g in gdf_dins_utm.geometry])
nn_dists_dins, nn_idx_dins = tree_fp.query(dins_pts, k=1)

# ── Compute per-structure-class median footprint area from matched buildings ──
struct_areas = {}   # STRUCTUREC -> [area_m2, ...]
for i, row in enumerate(gdf_dins_utm.itertuples()):
    if nn_dists_dins[i] <= MATCH_M:
        area = gdf_fp_utm.iloc[nn_idx_dins[i]]['area_m2']
        struct_areas.setdefault(row.STRUCTUREC, []).append(area)

struct_medians = {cls: float(np.median(vals)) for cls, vals in struct_areas.items()}
OVERALL_MEDIAN = float(np.median([a for vals in struct_areas.values() for a in vals]))

print(f"Match threshold : {MATCH_M} m")
print(f"Matched         : {(nn_dists_dins <= MATCH_M).sum():,}")
print(f"Unmatched       : {(nn_dists_dins > MATCH_M).sum():,}")
print()
print("Median footprint area by structure class (from matched buildings):")
for cls, med in sorted(struct_medians.items(), key=lambda x: -x[1]):
    n = len(struct_areas[cls])
    print(f"  {cls:<35} {med:>8.0f} m²  (n={n:,})")
print(f"  {'Overall fallback':<35} {OVERALL_MEDIAN:>8.0f} m²")


In [ ]:
# ── Assemble unified dataset ──────────────────────────────────────────
records   = []
n_matched = 0
n_est     = 0

for i, row in enumerate(gdf_dins_utm.itertuples()):
    dist  = nn_dists_dins[i]
    fp_i  = nn_idx_dins[i]

    if dist <= MATCH_M:
        fp   = gdf_fp_utm.iloc[fp_i]
        geom = fp.geometry
        area = float(fp['area_m2'])
        geom_src  = 'matched_footprint'
        fp_source = fp['source']
        n_matched += 1
    else:
        med    = struct_medians.get(row.STRUCTUREC, OVERALL_MEDIAN)
        radius = np.sqrt(med / np.pi)
        geom   = row.geometry.buffer(radius)   # circular proxy
        area   = med
        geom_src  = 'estimated_centroid'
        fp_source = 'estimated'
        n_est += 1

    records.append({
        'geometry':        geom,
        'STRUCTUREC':      row.STRUCTUREC,
        'DAMAGE':          row.DAMAGE,
        'LATITUDE':        row.LATITUDE,
        'LONGITUDE':       row.LONGITUDE,
        'area_m2':         area,
        'fp_source':       fp_source,
        'geometry_source': geom_src,
        'nn_dist_m':       dist,
    })

gdf_unified = gpd.GeoDataFrame(records, crs='EPSG:32610')

print(f"Unified dataset  : {len(gdf_unified):,} structures")
print(f"  Real polygons  : {n_matched:,}  ({n_matched/len(gdf_unified)*100:.1f}%)")
print(f"  Circular est.  : {n_est:,}  ({n_est/len(gdf_unified)*100:.1f}%)")
print()
print("Area summary (all structures):")
print(f"  Median : {gdf_unified['area_m2'].median():.0f} m²")
print(f"  Mean   : {gdf_unified['area_m2'].mean():.0f} m²")
print(f"  IQR    : {gdf_unified['area_m2'].quantile(0.25):.0f} – "
      f"{gdf_unified['area_m2'].quantile(0.75):.0f} m²")


In [ ]:
# ── Nearest-neighbour spacing: original vs unified ────────────────────
def nn_spacing(gdf_utm):
    """Return centroid-to-centroid nearest-neighbour distances (m)."""
    pts = np.array([[g.centroid.x, g.centroid.y] for g in gdf_utm.geometry])
    d, _ = cKDTree(pts).query(pts, k=2)
    return d[:, 1]

nn_original = nn_spacing(gdf_fp_utm[gdf_fp_utm['area_m2'] >= 15])
nn_unified  = nn_spacing(gdf_unified[gdf_unified['area_m2'] >= 15])

print("=== Nearest-Neighbour Spacing Comparison (metres) ===")
print(f"{'Percentile':<12} {'Original (12,933)':>20} {'Unified (17,964)':>20}")
print("-" * 54)
for p in [10, 25, 50, 75, 90]:
    print(f"  {p:3d}th       {np.percentile(nn_original, p):>18.1f} m "
          f"{np.percentile(nn_unified, p):>18.1f} m")

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
clip = 120

for ax, dists, label, color, n in [
    (axes[0], nn_original, f'Original ML dataset
({len(nn_original):,} buildings)', 'coral',    len(nn_original)),
    (axes[1], nn_unified,  f'DINS-centric unified
({len(nn_unified):,} structures)', '#2196F3', len(nn_unified)),
]:
    ax.hist(dists[dists < clip], bins=60, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.median(dists), color='darkred', linestyle='--', linewidth=1.5,
               label=f'Median: {np.median(dists):.1f} m')
    ax.set_xlabel('Distance to nearest centroid (m)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(label, fontsize=10)
    ax.legend(fontsize=9)

fig.suptitle('Building Spacing — Camp Fire Area (Paradise, CA)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(DATA_DIR / 'paradise_spacing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Map: matched footprints vs estimated centroids (in WGS84) ────────
gdf_unified_wgs = gdf_unified.to_crs('EPSG:4326')

matched_mask_map  = gdf_unified_wgs['geometry_source'] == 'matched_footprint'
estimated_mask_map = ~matched_mask_map

fig, ax = plt.subplots(figsize=(14, 9))

gdf_unified_wgs[estimated_mask_map].plot(
    ax=ax, color='#FF6B35', edgecolor='none', alpha=0.55,
    label=f"Estimated circular ({estimated_mask_map.sum():,})")
gdf_unified_wgs[matched_mask_map].plot(
    ax=ax, color='#2196F3', edgecolor='none', alpha=0.75,
    label=f"Matched polygon ({matched_mask_map.sum():,})")

gpd.GeoSeries([fire_perimeter]).plot(
    ax=ax, facecolor='none', edgecolor='black',
    linewidth=1, linestyle='--', label='Camp Fire perimeter')

ax.legend(fontsize=9, loc='upper right')
ax.set_title(
    'Unified Pre-Fire Structure Dataset — Camp Fire Area\n'
    'Blue = real polygon footprint  ·  Orange = estimated circular (DINS centroid + median area)',
    fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(DATA_DIR / 'paradise_unified_geometry_source.png', dpi=150, bbox_inches='tight')
plt.show()

# Save unified dataset
unified_path = DATA_DIR / 'camp_fire_unified_structures.gpkg'
gdf_unified.to_crs('EPSG:4326').to_file(unified_path, driver='GPKG')
print(f"Unified dataset saved → {unified_path}")


---
## 11. General Method: Pre-Fire Structure Inventory Without DINS

The DINS-centric approach above requires CAL FIRE post-fire inspection data, which only
exists for California fires. This section implements the same hybrid workflow using
**county parcel / tax-assessor data** as the structure inventory — a source available
for virtually every US county and fully pre-fire by definition.

### Why parcel data works

Tax-assessor records are created when a structure is *built*, not when it is *destroyed*.
Even after a fire, parcel records remain on the books (for tax purposes) until the property
is reassessed — often years later. Critically, the parcel attributes (year built, square
footage, structure type) describe the *pre-fire* structure.

### Data access pattern

Most US counties expose parcel data through an ArcGIS REST FeatureServer endpoint:

```
https://<county-gis-host>/arcgis/rest/services/<service>/FeatureServer/0/query
  ?geometry={"xmin":...,"xmax":...,"ymin":...,"ymax":...}
  &geometryType=esriGeometryEnvelope
  &inSR=4326&spatialRel=esriSpatialRelIntersects
  &outFields=APN,SITEADDR,YEARBUILT,SQFT,LANDUSE
  &f=geojson
  &resultRecordCount=2000
```

**Butte County (Paradise) URL:**
`https://gis.buttecounty.net/arcgis/rest/services/Parcels/MapServer/0/query`

**Finding any county's URL:**  
1. Search `"<County Name> GIS ArcGIS REST parcels"`  
2. Or use https://opendata.arcgis.com — most counties publish parcels as open datasets  
3. Or use the Regrid API (free tier covers light research use)

### Fallback chain (implemented below)

```
1. County ArcGIS REST endpoint  (most common, works for ~80% of US counties)
2. State-level parcel portal     (CA: CALPADS; TX: TCEQ; etc.)
3. OpenAddresses.io              (address points + footprint area where available)
4. OSM building density proxy    (use ohsome counts in intact adjacent area to estimate
                                  missing building count in burned area — last resort)
```


In [ ]:
# ── General parcel inventory fetcher ─────────────────────────────────
# Swap in the appropriate county URL for any US fire event.
# Returns a GeoDataFrame with standardised columns matching the DINS schema.

def fetch_parcel_inventory(
    bbox,                   # (xmin, ymin, xmax, ymax) WGS84
    county_arcgis_url,      # ArcGIS FeatureServer query endpoint
    sqft_field  = 'SQFT',   # field name for building square footage
    year_field  = 'YEARBUILT',
    class_field = 'LANDUSE',
    max_records = 5000,
    timeout     = 60,
):
    """
    Fetch building-level parcel records from an ArcGIS REST endpoint.

    Returns
    -------
    GeoDataFrame with columns:
        geometry    - parcel polygon (WGS84)
        STRUCTUREC  - structure class (mapped from LANDUSE)
        sqft        - recorded building square footage
        year_built  - year structure was constructed
        area_m2     - sqft converted to m²
        source      - 'parcel'
    """
    xmin, ymin, xmax, ymax = bbox
    params = {
        'geometry':         f'{{"xmin":{xmin},"ymin":{ymin},"xmax":{xmax},"ymax":{ymax}}}',
        'geometryType':     'esriGeometryEnvelope',
        'inSR':             '4326',
        'spatialRel':       'esriSpatialRelIntersects',
        'outFields':        ','.join([sqft_field, year_field, class_field,
                                      'APN', 'SITEADDR']),
        'where':            f"{year_field} > 0",   # exclude unimproved lots
        'returnGeometry':   'true',
        'outSR':            '4326',
        'f':                'geojson',
        'resultRecordCount': max_records,
    }

    resp = requests.get(county_arcgis_url, params=params, timeout=timeout)
    resp.raise_for_status()
    fc   = resp.json()
    feats = fc.get('features', [])
    if not feats:
        print("  No features returned — check URL or field names.")
        return gpd.GeoDataFrame()

    # LANDUSE → STRUCTUREC mapping (customise per county schema)
    LANDUSE_MAP = {
        'single':     'Single Residence',
        'sfr':        'Single Residence',
        'res':        'Single Residence',
        'multi':      'Multiple Residence',
        'mfr':        'Multiple Residence',
        'apt':        'Multiple Residence',
        'comm':       'Nonresidential Commercial',
        'com':        'Nonresidential Commercial',
        'retail':     'Nonresidential Commercial',
        'office':     'Nonresidential Commercial',
        'industrial': 'Nonresidential Commercial',
        'mixed':      'Mixed Commercial/Residential',
    }

    records = []
    for f in feats:
        props = f.get('properties', {}) or {}
        geom_raw = f.get('geometry')
        if geom_raw is None:
            continue
        try:
            geom = shape(geom_raw)
        except Exception:
            continue
        if geom.is_empty or not geom.is_valid:
            geom = geom.buffer(0)

        sqft       = props.get(sqft_field) or 0
        year_built = props.get(year_field)
        landuse    = str(props.get(class_field, '')).lower()

        # Map land use to STRUCTUREC
        structurec = 'Single Residence'   # default
        for key, label in LANDUSE_MAP.items():
            if key in landuse:
                structurec = label
                break

        records.append({
            'geometry':   geom,
            'STRUCTUREC': structurec,
            'sqft':       float(sqft),
            'year_built': year_built,
            'area_m2':    float(sqft) * 0.0929,   # ft² → m²
            'source':     'parcel',
            'LATITUDE':   geom.centroid.y,
            'LONGITUDE':  geom.centroid.x,
        })

    gdf = gpd.GeoDataFrame(records, crs='EPSG:4326')
    print(f"  Fetched {len(gdf):,} parcels with structures")
    return gdf


# ── Attempt to fetch Butte County parcels (requires network access) ───
BUTTE_PARCEL_URL = (
    "https://gis.buttecounty.net/arcgis/rest/services/Parcels/MapServer/0/query"
)

print("Attempting Butte County parcel fetch...")
print("(This cell requires direct network access to gis.buttecounty.net.)")
print("If blocked, download the parcel layer from the county GIS portal and")
print("load it directly: gpd.read_file('butte_parcels.geojson')")
print()

try:
    gdf_parcels = fetch_parcel_inventory(
        bbox             = BBOX,
        county_arcgis_url= BUTTE_PARCEL_URL,
        sqft_field       = 'SQFT',
        year_field       = 'YEARBUILT',
        class_field      = 'LANDUSE',
    )
    PARCEL_AVAILABLE = len(gdf_parcels) > 0
    if PARCEL_AVAILABLE:
        print(gdf_parcels[['STRUCTUREC', 'sqft', 'year_built', 'area_m2']].head(5))
except Exception as e:
    print(f"  Parcel fetch failed: {e}")
    print()
    print("  → To use parcel-based inventory:")
    print("    1. Download parcel shapefile from https://www.buttecounty.net/publicworks/GIS")
    print("    2. gdf_parcels = gpd.read_file('ButteParcels.shp')")
    print("    3. Filter: gdf_parcels = gdf_parcels[gdf_parcels['YEARBUILT'] < 2018]")
    print("    4. Re-run the cell below.")
    PARCEL_AVAILABLE = False


In [ ]:
# ── Convert parcel inventory → unified hybrid dataset ─────────────────
# Exactly the same logic as Section 10, but using parcel data instead of DINS.
# Parcel polygons ARE the lot boundaries (not building footprints), so we use:
#   - Parcel centroid as the building location
#   - Parcel SQFT (tax-recorded building area) to size a circular footprint
# This produces the same geometry_source='estimated_centroid' result, but driven
# entirely by pre-fire tax records rather than post-fire damage inspection.
#
# Where OSM has a building polygon inside the parcel (matched via centroid KDTree),
# we prefer the OSM polygon — same priority logic as the DINS hybrid above.

def parcel_to_unified(gdf_parcels_wgs84, gdf_osm_utm, match_m=25):
    """
    Build a unified structure GeoDataFrame from parcel records.

    Priority:
      1. If an OSM building centroid is within match_m of the parcel centroid → use OSM polygon
      2. Otherwise → circular footprint sized from parcel SQFT (or class median if SQFT = 0)

    Returns GeoDataFrame (EPSG:32610) with geometry_source column.
    """
    gdf_p = gdf_parcels_wgs84.to_crs('EPSG:32610').copy()
    gdf_p['area_m2'] = gdf_p['area_m2'].where(gdf_p['area_m2'] > 5, np.nan)

    # Class median areas from OSM reference pool
    osm_ref_median = gdf_osm_utm['area_m2'].median() if 'area_m2' in gdf_osm_utm.columns else 100.0

    # KDTree on OSM centroids
    if len(gdf_osm_utm) > 0:
        osm_pts = np.array([[g.centroid.x, g.centroid.y] for g in gdf_osm_utm.geometry])
        tree_osm = cKDTree(osm_pts)
    else:
        tree_osm = None

    parcel_centroids = np.array([[g.centroid.x, g.centroid.y] for g in gdf_p.geometry])

    records = []
    n_osm_match = 0

    for i, row in enumerate(gdf_p.itertuples()):
        cx, cy = parcel_centroids[i]
        from shapely.geometry import Point
        centroid_geom = Point(cx, cy)

        if tree_osm is not None:
            dist, osm_i = tree_osm.query([[cx, cy]], k=1)
            dist = float(dist[0])
        else:
            dist, osm_i = 999, 0

        if dist <= match_m:
            geom     = gdf_osm_utm.iloc[osm_i].geometry
            area     = float(gdf_osm_utm.iloc[osm_i]['area_m2'])
            geom_src = 'matched_footprint'
            fp_src   = 'openstreetmap'
            n_osm_match += 1
        else:
            area = row.area_m2 if not np.isnan(row.area_m2) else osm_ref_median
            radius   = np.sqrt(area / np.pi)
            geom     = centroid_geom.buffer(radius)
            geom_src = 'estimated_centroid'
            fp_src   = 'parcel'

        records.append({
            'geometry':        geom,
            'STRUCTUREC':      row.STRUCTUREC,
            'area_m2':         area,
            'fp_source':       fp_src,
            'geometry_source': geom_src,
            'LATITUDE':        row.LATITUDE,
            'LONGITUDE':       row.LONGITUDE,
        })

    gdf_out = gpd.GeoDataFrame(records, crs='EPSG:32610')
    pct_osm = n_osm_match / max(len(gdf_out), 1) * 100
    print(f"Parcel-based unified dataset: {len(gdf_out):,} structures")
    print(f"  OSM polygon match : {n_osm_match:,}  ({pct_osm:.1f}%)")
    print(f"  Parcel estimate   : {len(gdf_out) - n_osm_match:,}  ({100-pct_osm:.1f}%)")
    return gdf_out


if PARCEL_AVAILABLE:
    gdf_osm_utm_ref = gdf_osm.to_crs('EPSG:32610').copy()
    gdf_osm_utm_ref['area_m2'] = gdf_osm_utm_ref.geometry.area
    gdf_parcel_unified = parcel_to_unified(gdf_parcels, gdf_osm_utm_ref, match_m=25)
    print()
    print("Run the spacing / orientation cells on gdf_parcel_unified to get")
    print("the same spatial metrics as Section 10 — but driven purely by pre-fire")
    print("tax records, with no dependence on post-fire damage inspection data.")
else:
    print("Parcel data not available — skipping parcel-based unified dataset.")
    print("Re-run this cell after loading gdf_parcels from a local file.")
    print()
    print("Inventory source decision tree for non-CA fires:")
    print("  1. County ArcGIS parcel REST  → fetch_parcel_inventory()")
    print("  2. State parcel open data     → gpd.read_file('parcels.gpkg')")
    print("  3. FEMA NSI (when accessible) → https://nsi.sec.usace.army.mil/nsiapi/structures?bbox=...")
    print("  4. Regrid API (light research)→ https://app.regrid.com/api")
    print("  5. OpenAddresses.io           → https://batch.openaddresses.io/")


---
## 8. Save Output

Output GeoPackage includes all footprints with:
- `source` — `openstreetmap` or `microsoft-buildings` (or other ML source)
- `source_date` — OSM last-edit timestamp (NaT for ML sources)
- `area_m2` — footprint area (projected, UTM Zone 10N)
- `orientation_deg` — long-axis orientation angle (MRR, 0–180°)
- `min_wall_wall_m` — **minimum wall-to-wall gap to nearest polygon neighbour** (primary separation metric)
- `nn_dist_m` — centroid-to-centroid distance to nearest neighbour (reference)


In [ ]:
# Merge spatial metrics back into WGS84 GDF for saving
metric_cols = ['area_m2', 'orientation_deg', 'nn_dist_m']

# gdf_main is projected — reproject back and transfer metrics
gdf_out = gdf_all.copy()
gdf_out = gdf_out.merge(
    gdf_main[metric_cols].reset_index(),
    left_index=True, right_on='index', how='left'
).drop(columns='index', errors='ignore')

# Columns to save (drop colour and internal fields)
save_cols = ['geometry', 'source', 'source_date', 'osm_id', 'osm_type',
             'building', 'name', 'height', 'levels',
             'addr_street', 'addr_housenumber',
             'area_m2', 'orientation_deg', 'min_wall_wall_m', 'nn_dist_m']
save_cols = [c for c in save_cols if c in gdf_out.columns]

gdf_save = gpd.GeoDataFrame(gdf_out[save_cols], crs='EPSG:4326')

# GeoPackage (best for attribute-rich data)
gpkg_path = DATA_DIR / 'camp_fire_prefire_buildings.gpkg'
gdf_save.to_file(gpkg_path, driver='GPKG')
print(f"Saved {len(gdf_save):,} buildings → {gpkg_path}")

# GeoJSON (for web / GitHub)
geojson_path = DATA_DIR / 'camp_fire_prefire_buildings.geojson'
gdf_save.to_file(geojson_path, driver='GeoJSON')
print(f"Saved → {geojson_path}")

# Summary CSV (no geometry)
csv_path = DATA_DIR / 'camp_fire_prefire_buildings_summary.csv'
gdf_save.drop(columns='geometry').to_csv(csv_path, index=False)
print(f"Saved → {csv_path}")

In [ ]:
# Final summary
print("=" * 50)
print(" CAMP FIRE AREA — PRE-FIRE BUILDING FOOTPRINTS")
print("=" * 50)
print(f" Query date      : {OSM_DATE}")
print(f" Total buildings : {len(gdf_save):,}")
print()
for src, grp in gdf_save.groupby('source'):
    has_date = grp['source_date'].notna().sum()
    print(f"  {src:<20} {len(grp):>5,} buildings  "
          f"({has_date:,} with source_date)")
print()
print(" Output files:")
for p in [gpkg_path, geojson_path, csv_path]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<45} {size_kb:>8.0f} KB")